In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for UTC+8 as required
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrames (keep full tables in memory as variables)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Ensure numeric values for metric/trace/log value columns
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) metric.csv aggregation: per (cmdb_id, kpi_name) count and global P95
_metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
_metric_count = _metric_grp.size().rename('count')
_metric_p95 = _metric_grp.quantile(0.95).rename('p95_value')
metric_agg = pd.concat([_metric_count, _metric_p95], axis=1).reset_index()
metric_agg = metric_agg.sort_values('count', ascending=False).head(50)

# 2) trace.csv aggregation: per (cmdb_id, trace_name) count and global P95
_trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
_trace_count = _trace_grp.size().rename('count')
_trace_p95 = _trace_grp.quantile(0.95).rename('p95_value')
trace_agg = pd.concat([_trace_count, _trace_p95], axis=1).reset_index()
trace_agg = trace_agg.sort_values('count', ascending=False).head(50)

# 3) log.csv aggregation: per (cmdb_id, log_name) count and global P95
_log_grp = log_df.groupby(['cmdb_id', 'log_name'])['value']
_log_count = _log_grp.size().rename('count')
_log_p95 = _log_grp.quantile(0.95).rename('p95_value')
log_agg = pd.concat([_log_count, _log_p95], axis=1).reset_index()
log_agg = log_agg.sort_values('count', ascending=False).head(50)

# 4) error_logs.csv aggregation: per cmdb_id count, earliest and latest timestamp (converted to UTC+8)
# convert timestamp seconds to timezone-aware datetimes in UTC then to Asia/Shanghai
error_df['ts'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
_error_grp = error_df.groupby('cmdb_id')['ts']
error_count = _error_grp.size().rename('count')
error_earliest = _error_grp.min().rename('earliest_ts')
error_latest = _error_grp.max().rename('latest_ts')
error_agg = pd.concat([error_count, error_earliest, error_latest], axis=1).reset_index()
# Format timestamps as compact strings including timezone offset
error_agg['earliest_ts'] = error_agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
error_agg['latest_ts'] = error_agg['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
error_agg = error_agg.sort_values('count', ascending=False).head(50)

# Keep outputs compact: show only requested aggregate tables (top 50 where applicable)
metric_agg, trace_agg, log_agg, error_agg

```
Out[1]:
```
```python
# Compose a compact plain-English summary based on previously computed aggregates.
# Reuse existing variables: metric_agg, trace_agg, log_agg, error_agg, metric_df, trace_df, log_df, error_df

# counts of unique groups in full files
metric_groups = metric_df.groupby(['cmdb_id', 'kpi_name']).size().shape[0]
trace_groups = trace_df.groupby(['cmdb_id', 'trace_name']).size().shape[0]
log_groups = log_df.groupby(['cmdb_id', 'log_name']).size().shape[0]
error_rows = error_df.shape[0]

# helper to format top rows compactly
def top_rows_summary(df, key1, key2, topn=5):
    rows = []
    for _, r in df.head(topn).iterrows():
        rows.append(f"{r[key1]} | {r[key2]} (count={int(r['count'])}, p95={r['p95_value']:.6g})")
    return rows

metric_top = top_rows_summary(metric_agg, 'cmdb_id', 'kpi_name', topn=5)
trace_top = top_rows_summary(trace_agg, 'cmdb_id', 'trace_name', topn=5)
log_top = top_rows_summary(log_agg, 'cmdb_id', 'log_name', topn=5)

# error_logs summary: check if any rows/groups
if error_rows == 0 or error_agg.empty:
    error_summary = "No error_log entries found in error_logs.csv."
else:
    # summarize top cmdb_ids by count
    ers = []
    for _, r in error_agg.head(5).iterrows():
        ers.append(f"{r['cmdb_id']} (count={int(r['count'])}, earliest={r['earliest_ts']}, latest={r['latest_ts']})")
    error_summary = "Top error log sources: " + "; ".join(ers)

summary = (
    "Summary of telemetry aggregates (compact):\n\n"
    f"1) metric.csv: {metric_groups} unique (cmdb_id, kpi_name) groups.\n"
    "   Top groups (by row count, top 5):\n   - " + "\n   - ".join(metric_top) + "\n\n"
    f"2) trace.csv: {trace_groups} unique (cmdb_id, trace_name) groups.\n"
    "   Top groups (by row count, top 5):\n   - " + "\n   - ".join(trace_top) + "\n\n"
    f"3) log.csv: {log_groups} unique (cmdb_id, log_name) groups.\n"
    "   Top groups (by row count, top 5):\n   - " + "\n   - ".join(log_top) + "\n\n"
    f"4) error_logs.csv: total rows = {error_rows}.\n"
    f"   {error_summary}\n\n"
    "Notes: p95 values shown are the global P95 computed across the entire file for each group."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count     p95_value
1577  apache01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.500000e+01
1576  apache01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00
1591  apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.522187e+06
1659  apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00
...        ...                                                ...    ...           ...
182       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed     31  1.038978e+09
103       IG02                         OSLinux-CPU_CPU_CPUSysTime     31  4.322000e-01
1383  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  3.303510e+05
171       IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  3.000000e-01
172       IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  3.000000e+00

[50 rows x 4 columns],       cmdb_id                       trace_name  count  p95_value
17       IG02             trace.self.row_count     26  39.000000
15       IG02         trace.self.duration_mean     26   0.129872
16       IG02          trace.self.duration_p95     26   0.447388
93       MG02         trace.self.duration_mean     26   0.048850
94       MG02          trace.self.duration_p95     26   0.126850
..        ...                              ...    ...        ...
151  Tomcat03       trace.to_MG02.duration_p95     20   0.113863
149  Tomcat03          trace.to_MG01.row_count     20   4.250000
5        IG01      trace.to_Tomcat01.row_count     20   6.150000
13       IG01   trace.to_Tomcat04.duration_p95     20   0.317643
3        IG01  trace.to_Tomcat01.duration_mean     20   0.243525

[50 rows x 4 columns],      cmdb_id          log_name  count  p95_value
10  Tomcat03     log.row_count     24    10.0000
8   Tomcat03   log.error_count     24     0.0000
16  apache02   log.error_count     23     0.0000
15  apache01     log.row_count     23    18.5000
5   Tomcat02   log.error_count     23     0.0000
2   Tomcat01   log.error_count     23     0.0000
7   Tomcat02     log.row_count     23    10.0000
13  Tomcat04     log.row_count     23     8.0000
11  Tomcat04   log.error_count     23     0.0000
4   Tomcat01     log.row_count     23    10.7000
17  apache02     log.row_count     23    20.5000
14  apache01   log.error_count     23     0.0000
6   Tomcat02  log.gc_pause_max      2     0.0695
0    Mysql01   log.error_count      1     0.0000
1    Mysql01     log.row_count      1     8.0000
3   Tomcat01  log.gc_pause_max      1     0.1300
9   Tomcat03  log.gc_pause_max      1     0.0600
12  Tomcat04  log.gc_pause_max      1     0.0800, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, latest_ts]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing variables: metric_df, metric_agg
tz = pytz.timezone('Asia/Shanghai')

# 1) Convert timestamp to tz-aware datetimes (UTC->Asia/Shanghai) and filter to the window
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start_dt = tz.localize(pd.to_datetime('2021-03-10 04:30:00'))
end_dt = tz.localize(pd.to_datetime('2021-03-10 05:00:00'))
metric_window = metric_df[(metric_df['ts'] >= start_dt) & (metric_df['ts'] <= end_dt)].copy()

# 2) Join with precomputed global p95 (metric_agg) on (cmdb_id, kpi_name)
# ensure metric_agg has expected columns
if 'p95_value' not in metric_agg.columns:
    raise ValueError("metric_agg missing p95_value column")
merged = metric_window.merge(metric_agg[['cmdb_id', 'kpi_name', 'p95_value']],
                             on=['cmdb_id', 'kpi_name'], how='left')

# 3) Identify anomaly points where value > p95_value. Skip groups with NaN p95_value.
merged = merged.dropna(subset=['p95_value']).copy()
merged['is_anom'] = merged['value'] > merged['p95_value']

# 4) For each group in the window, compute required aggregates but only for groups with anomalies
anom_rows = merged[merged['is_anom']]
if anom_rows.empty:
    anomaly_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value'])
else:
    grp = anom_rows.groupby(['cmdb_id', 'kpi_name'])
    summary = grp.agg(
        anomaly_count=('is_anom', 'size'),
        earliest_ts=('ts', 'min'),
        latest_ts=('ts', 'max'),
        max_value=('value', 'max')
    ).reset_index()
    # bring p95_value (global) per group
    p95_per_group = merged.groupby(['cmdb_id', 'kpi_name'])['p95_value'].first().reset_index()
    anomaly_summary = summary.merge(p95_per_group, on=['cmdb_id', 'kpi_name'], how='left')
    # format timestamps compactly as strings in UTC+8
    anomaly_summary['earliest_ts'] = anomaly_summary['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    anomaly_summary['latest_ts'] = anomaly_summary['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 5) Return compact table with requested columns, top 20 by anomaly_count desc then max_value desc
anomaly_summary = anomaly_summary[['cmdb_id','kpi_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value']]
anomaly_summary = anomaly_summary.sort_values(['anomaly_count','max_value'], ascending=[False, False]).head(20)

anomaly_summary

```
Out[2]:
```
```python
# Summarize anomaly detection results in plain English using existing anomaly_summary DataFrame.
# Reuse kernel variable 'anomaly_summary' computed earlier.

total_groups = anomaly_summary.shape[0]
top_n = min(10, total_groups)
top_rows = anomaly_summary.head(top_n)

lines = [f"Metric anomaly detection summary (window 2021-03-10 04:30–05:00 UTC+8):",
         f"- Total groups with anomalies: {total_groups} (only groups with value > global P95 shown).",
         f"- Showing top {top_n} groups by anomaly_count then max_value:"]

for _, r in top_rows.iterrows():
    lines.append(
        f"  • {r['cmdb_id']} | {r['kpi_name']} — anomalies={int(r['anomaly_count'])}, "
        f"earliest={r['earliest_ts']}, latest={r['latest_ts']}, "
        f"max_value={r['max_value']:.6g}, p95={r['p95_value']:.6g}"
    )

lines.append("\nInterpretation: Most anomalies are short (1–2 points) and occur around 04:59–05:00. "
             "Notable items include JVM heap usage spikes on MG01 and IG01 and several OS CPU/uptime metrics exceeding their global P95s.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                     kpi_name  anomaly_count               earliest_ts                 latest_ts     max_value     p95_value
9       MG01    JVM-Memory_7779_JVM_Memory_HeapMemoryUsed              2  2021-03-10 04:59:00+0800  2021-03-10 05:00:00+0800  1.043554e+09  1.038978e+09
0       IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed              2  2021-03-10 04:59:00+0800  2021-03-10 05:00:00+0800  3.999865e+08  3.974848e+08
10  apache01           OSLinux-OSLinux_ZABBIX_Host_Uptime              2  2021-03-10 04:59:00+0800  2021-03-10 05:00:00+0800  6.522277e+06  6.522187e+06
14  apache02           OSLinux-OSLinux_ZABBIX_Host_Uptime              2  2021-03-10 04:59:00+0800  2021-03-10 05:00:00+0800  6.522272e+06  6.522182e+06
13  apache02                  OSLinux-CPU_CPU_CPUidleutil              2  2021-03-10 04:35:00+0800  2021-03-10 04:59:00+0800  9.976320e+01  9.974871e+01
5       IG01                  OSLinux-CPU_CPU_CPUidleutil              2  2021-03-10 04:36:00+0800  2021-03-10 05:00:00+0800  7.468820e+01  7.466325e+01
8       IG02                  OSLinux-CPU_CPU_CPUUserTime              2  2021-03-10 04:34:00+0800  2021-03-10 04:44:00+0800  2.525248e+01  2.517195e+01
3       IG01                  OSLinux-CPU_CPU_CPUUserTime              2  2021-03-10 04:39:00+0800  2021-03-10 04:59:00+0800  2.520470e+01  2.515019e+01
1       IG01                      OSLinux-CPU_CPU_CPULoad              2  2021-03-10 04:31:00+0800  2021-03-10 04:33:00+0800  1.330000e+00  1.135000e+00
7       IG02                   OSLinux-CPU_CPU_CPUSysTime              2  2021-03-10 04:34:00+0800  2021-03-10 04:44:00+0800  5.527620e-01  4.322000e-01
11  apache02                   OSLinux-CPU_CPU_CPUSysTime              2  2021-03-10 04:48:00+0800  2021-03-10 04:58:00+0800  5.109000e-01  4.300505e-01
2       IG01                   OSLinux-CPU_CPU_CPUSysTime              2  2021-03-10 04:39:00+0800  2021-03-10 04:59:00+0800  4.739000e-01  4.157540e-01
4       IG01                       OSLinux-CPU_CPU_CPUWio              2  2021-03-10 04:30:00+0800  2021-03-10 04:48:00+0800  3.533000e-01  2.203000e-01
12  apache02                       OSLinux-CPU_CPU_CPUWio              2  2021-03-10 04:40:00+0800  2021-03-10 04:42:00+0800  2.699000e-01  1.453830e-01
6       IG02  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed              1  2021-03-10 04:56:00+0800  2021-03-10 04:56:00+0800  5.734566e+07  5.734074e+07```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing trace_df and trace_agg from kernel
tz = pytz.timezone('Asia/Shanghai')

# ensure numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Convert timestamp to UTC+8 tz-aware datetimes and filter to window
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start_dt = tz.localize(pd.to_datetime('2021-03-10 04:30:00'))
end_dt = tz.localize(pd.to_datetime('2021-03-10 05:00:00'))
trace_window = trace_df[(trace_df['ts'] >= start_dt) & (trace_df['ts'] <= end_dt)].copy()

# 2) Join with global p95 from trace_agg on (cmdb_id, trace_name); skip groups with NaN p95
merged_trace = trace_window.merge(trace_agg[['cmdb_id', 'trace_name', 'p95_value']],
                                  on=['cmdb_id', 'trace_name'], how='left')
merged_trace = merged_trace.dropna(subset=['p95_value']).copy()

# 3) Mark anomaly points where value > p95_value
merged_trace['is_anom'] = merged_trace['value'] > merged_trace['p95_value']

# 4) Aggregate per group for groups with anomalies
anom_points = merged_trace[merged_trace['is_anom']]
if anom_points.empty:
    trace_anomaly_summary = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value'])
else:
    agg = anom_points.groupby(['cmdb_id', 'trace_name']).agg(
        anomaly_count=('is_anom', 'size'),
        earliest_ts=('ts', 'min'),
        latest_ts=('ts', 'max'),
        max_value=('value', 'max')
    ).reset_index()
    # attach global p95_value per group
    p95_per_group = merged_trace.groupby(['cmdb_id', 'trace_name'])['p95_value'].first().reset_index()
    trace_anomaly_summary = agg.merge(p95_per_group, on=['cmdb_id', 'trace_name'], how='left')
    # format timestamps compactly
    trace_anomaly_summary['earliest_ts'] = trace_anomaly_summary['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    trace_anomaly_summary['latest_ts'] = trace_anomaly_summary['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 5) Select columns and sort/limit to top 20
trace_anomaly_summary = trace_anomaly_summary[
    ['cmdb_id','trace_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value']
].sort_values(['anomaly_count','max_value'], ascending=[False, False]).head(20)

trace_anomaly_summary

```
Out[3]:
```
```python
# Summarize trace anomaly detection results in plain English using existing trace_anomaly_summary variable.

total_groups = trace_anomaly_summary.shape[0]
top_n = min(10, total_groups)
top_rows = trace_anomaly_summary.head(top_n)

lines = [
    "Trace anomaly detection summary (window 2021-03-10 04:30–05:00 UTC+8):",
    f"- Total trace groups with anomalies: {total_groups} (value > global P95).",
    f"- Showing top {top_n} groups by anomaly_count then max_value:"
]

for _, r in top_rows.iterrows():
    lines.append(
        f"  • {r['cmdb_id']} | {r['trace_name']} — anomalies={int(r['anomaly_count'])}, "
        f"earliest={r['earliest_ts']}, latest={r['latest_ts']}, "
        f"max={r['max_value']:.6g}, p95={r['p95_value']:.6g}"
    )

lines.append(
    "\nInterpretation: Many anomalies are spikes in trace.self.row_count (request count) across services "
    "(dockerB1, Tomcat03/04, MG02, IG02, IG01) and spikes in trace durations (duration_p95/duration_mean) "
    "on MG01/MG02/IG01 and some Tomcat instances. These events mostly occur between ~04:37 and ~04:50 UTC+8, "
    "suggesting short-lived increases in load and elevated latencies during that interval."
)

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  anomaly_count               earliest_ts                 latest_ts   max_value   p95_value
45  dockerB1             trace.self.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:49:00+0800  154.000000  142.000000
32  Tomcat03             trace.self.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:40:00+0800   98.000000   65.700000
39  Tomcat04             trace.self.row_count              2  2021-03-10 04:40:00+0800  2021-03-10 04:47:00+0800   58.000000   55.600000
17      MG02             trace.self.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:40:00+0800   51.000000   42.000000
8       IG02             trace.self.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:40:00+0800   46.000000   39.000000
2       IG01             trace.self.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:40:00+0800   44.000000   39.600000
7       IG02          trace.self.duration_p95              2  2021-03-10 04:38:00+0800  2021-03-10 04:39:00+0800    0.497000    0.447388
13      MG01          trace.self.duration_p95              2  2021-03-10 04:37:00+0800  2021-03-10 04:39:00+0800    0.489400    0.214745
1       IG01          trace.self.duration_p95              2  2021-03-10 04:47:00+0800  2021-03-10 04:49:00+0800    0.344000    0.316470
19      MG02   trace.to_dockerB1.duration_p95              2  2021-03-10 04:40:00+0800  2021-03-10 04:49:00+0800    0.310000    0.131237
41  dockerB1     trace.from_MG02.duration_p95              2  2021-03-10 04:40:00+0800  2021-03-10 04:49:00+0800    0.310000    0.131237
18      MG02  trace.to_dockerB1.duration_mean              2  2021-03-10 04:49:00+0800  2021-03-10 04:50:00+0800    0.191667    0.113875
40  dockerB1    trace.from_MG02.duration_mean              2  2021-03-10 04:49:00+0800  2021-03-10 04:50:00+0800    0.191667    0.113875
22  Tomcat01          trace.self.duration_p95              2  2021-03-10 04:48:00+0800  2021-03-10 04:49:00+0800    0.174750    0.123180
16      MG02          trace.self.duration_p95              2  2021-03-10 04:49:00+0800  2021-03-10 04:50:00+0800    0.164250    0.126850
6       IG02         trace.self.duration_mean              2  2021-03-10 04:38:00+0800  2021-03-10 04:49:00+0800    0.156870    0.129872
0       IG01         trace.self.duration_mean              2  2021-03-10 04:40:00+0800  2021-03-10 04:49:00+0800    0.147917    0.120931
31  Tomcat03          trace.self.duration_p95              2  2021-03-10 04:40:00+0800  2021-03-10 04:49:00+0800    0.142850    0.078097
38  Tomcat04          trace.self.duration_p95              2  2021-03-10 04:37:00+0800  2021-03-10 04:49:00+0800    0.139900    0.088625
12      MG01         trace.self.duration_mean              2  2021-03-10 04:37:00+0800  2021-03-10 04:39:00+0800    0.079435    0.072467```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing kernel variables: log_df, log_agg
tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric and create timezone-aware timestamp column
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Filter to failure window (inclusive)
start_dt = tz.localize(pd.to_datetime('2021-03-10 04:30:00'))
end_dt = tz.localize(pd.to_datetime('2021-03-10 05:00:00'))
log_window = log_df[(log_df['ts'] >= start_dt) & (log_df['ts'] <= end_dt)].copy()

# 2) Join with global p95 from log_agg; skip groups where p95_value is NaN
merged_log = log_window.merge(log_agg[['cmdb_id', 'log_name', 'p95_value']],
                              on=['cmdb_id', 'log_name'], how='left')
merged_log = merged_log.dropna(subset=['p95_value']).copy()

# 3) Mark anomalies where value > p95_value
merged_log['is_anom'] = merged_log['value'] > merged_log['p95_value']

# 4) Aggregate per group for groups with anomalies
anom_points = merged_log[merged_log['is_anom']]
if anom_points.empty:
    log_anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','log_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value'
    ])
else:
    agg = anom_points.groupby(['cmdb_id', 'log_name']).agg(
        anomaly_count=('is_anom', 'size'),
        earliest_ts=('ts', 'min'),
        latest_ts=('ts', 'max'),
        max_value=('value', 'max')
    ).reset_index()
    # attach global p95_value per group
    p95_per_group = merged_log.groupby(['cmdb_id', 'log_name'])['p95_value'].first().reset_index()
    log_anomaly_summary = agg.merge(p95_per_group, on=['cmdb_id', 'log_name'], how='left')
    # format timestamps compactly
    log_anomaly_summary['earliest_ts'] = log_anomaly_summary['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    log_anomaly_summary['latest_ts'] = log_anomaly_summary['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 5) Select columns, sort and limit to top 20
log_anomaly_summary = log_anomaly_summary[
    ['cmdb_id','log_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value']
].sort_values(['anomaly_count','max_value'], ascending=[False, False]).head(20)

log_anomaly_summary

```
Out[4]:
```
```python
# Summarize log anomaly detection results in plain English using existing log_anomaly_summary variable.
total_groups = log_anomaly_summary.shape[0]
top_n = min(10, total_groups)
top_rows = log_anomaly_summary.head(top_n)

lines = [
    "Log anomaly detection summary (window 2021-03-10 04:30–05:00 UTC+8):",
    f"- Total log groups with anomalies: {total_groups} (value > global P95).",
    f"- Showing top {top_n} groups by anomaly_count then max_value:"
]

for _, r in top_rows.iterrows():
    lines.append(
        f"  • {r['cmdb_id']} | {r['log_name']} — anomalies={int(r['anomaly_count'])}, "
        f"earliest={r['earliest_ts']}, latest={r['latest_ts']}, "
        f"max_value={r['max_value']:.6g}, p95={r['p95_value']:.6g}"
    )

lines.append(
    "\nInterpretation: Short-lived spikes in log.row_count occurred primarily on apache02, apache01 and Tomcat01 around 04:38–04:40 UTC+8 "
    "(counts exceeded their global P95s). A smaller spike occurred on Tomcat03/02/04. Additionally, Tomcat02 had one log.gc_pause_max slightly above its P95 at 04:43. "
    "These are brief bursts rather than sustained elevated log rates."
)

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  anomaly_count               earliest_ts                 latest_ts  max_value  p95_value
6  apache02     log.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:40:00+0800      24.00    20.5000
5  apache01     log.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:40:00+0800      23.00    18.5000
0  Tomcat01     log.row_count              2  2021-03-10 04:38:00+0800  2021-03-10 04:40:00+0800      14.00    10.7000
3  Tomcat03     log.row_count              1  2021-03-10 04:38:00+0800  2021-03-10 04:38:00+0800      15.00    10.0000
2  Tomcat02     log.row_count              1  2021-03-10 04:38:00+0800  2021-03-10 04:38:00+0800      12.00    10.0000
4  Tomcat04     log.row_count              1  2021-03-10 04:40:00+0800  2021-03-10 04:40:00+0800       9.00     8.0000
1  Tomcat02  log.gc_pause_max              1  2021-03-10 04:43:00+0800  2021-03-10 04:43:00+0800       0.07     0.0695```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variable trace_anomaly_summary from kernel
# Ensure it exists
try:
    df = trace_anomaly_summary.copy()
except NameError:
    raise NameError("trace_anomaly_summary not found in kernel. Run trace anomaly detection step first.")

# Compute severity_ratio = max_value / p95_value, handling zero/NaN p95_value safely
df['p95_value'] = pd.to_numeric(df['p95_value'], errors='coerce')
df['max_value'] = pd.to_numeric(df['max_value'], errors='coerce')
df['severity_ratio'] = df['max_value'] / df['p95_value']
# Where p95_value is zero, set severity to inf to indicate full breach
df.loc[df['p95_value'] == 0, 'severity_ratio'] = np.inf

# Parse earliest_ts for sorting (preserve original string columns for output)
df['earliest_dt'] = pd.to_datetime(df['earliest_ts'], utc=True)

# Filter groups meeting severity >= 1.5
severe = df[df['severity_ratio'] >= 1.5].copy()

if not severe.empty:
    # Sort by earliest timestamp ascending and limit to top 20
    result_df = severe.sort_values('earliest_dt', ascending=True).head(20)
else:
    # If none meet threshold, return top 20 by severity_ratio desc
    result_df = df.sort_values('severity_ratio', ascending=False).head(20)

# Keep only requested columns and format numeric columns compactly
result_df = result_df[
    ['cmdb_id','trace_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value','severity_ratio']
].reset_index(drop=True)

# Round severity_ratio and numeric outputs for compactness
result_df['severity_ratio'] = result_df['severity_ratio'].round(6)
result_df['max_value'] = result_df['max_value'].round(6)
result_df['p95_value'] = result_df['p95_value'].round(6)

result_df

```
Out[5]:
```
```python
# Summarize the high-severity trace anomalies (severity_ratio >= 1.5) from existing result_df
# (result_df was produced in the previous step and contains the filtered/ordered groups).

df = result_df.copy()  # reuse kernel variable
count = df.shape[0]

lines = [
    f"Trace severity summary (window 2021-03-10 04:30–05:00 UTC+8):",
    f"- Groups meeting severity_ratio >= 1.5: {count}",
]
for _, r in df.iterrows():
    lines.append(
        f"  • {r['cmdb_id']} | {r['trace_name']} — anomalies={int(r['anomaly_count'])}, "
        f"earliest={r['earliest_ts']}, latest={r['latest_ts']}, "
        f"max={r['max_value']}, p95={r['p95_value']}, severity={r['severity_ratio']}"
    )

lines.append(
    "\nInterpretation: Highest relative breaches are on MG02 <-> dockerB1 duration_p95 (severity ~2.36) "
    "and MG01 trace.self.duration_p95 (severity ~2.28). These indicate short-lived but substantial latency spikes "
    "affecting MG↔dockerB1 paths and some service p95 latencies (Tomcat/MG) between ~04:37 and ~04:50 UTC+8."
)

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  anomaly_count               earliest_ts                 latest_ts  max_value  p95_value  severity_ratio
0      MG01          trace.self.duration_p95              2  2021-03-10 04:37:00+0800  2021-03-10 04:39:00+0800   0.489400   0.214745        2.278982
1  Tomcat04          trace.self.duration_p95              2  2021-03-10 04:37:00+0800  2021-03-10 04:49:00+0800   0.139900   0.088625        1.578561
2      MG02   trace.to_dockerB1.duration_p95              2  2021-03-10 04:40:00+0800  2021-03-10 04:49:00+0800   0.310000   0.131237        2.362130
3  dockerB1     trace.from_MG02.duration_p95              2  2021-03-10 04:40:00+0800  2021-03-10 04:49:00+0800   0.310000   0.131237        2.362130
4  Tomcat03          trace.self.duration_p95              2  2021-03-10 04:40:00+0800  2021-03-10 04:49:00+0800   0.142850   0.078097        1.829124
5      MG02  trace.to_dockerB1.duration_mean              2  2021-03-10 04:49:00+0800  2021-03-10 04:50:00+0800   0.191667   0.113875        1.683135
6  dockerB1    trace.from_MG02.duration_mean              2  2021-03-10 04:49:00+0800  2021-03-10 04:50:00+0800   0.191667   0.113875        1.683135```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse trace_df and trace_agg from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df has timezone-aware 'ts' column (create if missing)
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and correct tz
    trace_df['ts'] = pd.to_datetime(trace_df['ts']).dt.tz_localize(None, ambiguous='infer').dt.tz_localize('UTC').dt.tz_convert(tz) if not pd.api.types.is_datetime64tz_dtype(trace_df['ts']) else trace_df['ts'].dt.tz_convert(tz)

# Ensure numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Define window (inclusive)
start_dt = tz.localize(pd.to_datetime('2021-03-10 04:30:00'))
end_dt = tz.localize(pd.to_datetime('2021-03-10 05:00:00'))

# 1) Filter trace_df to the time window
trace_window = trace_df[(trace_df['ts'] >= start_dt) & (trace_df['ts'] <= end_dt)].copy()

# 2) Join with trace_agg to get p95_value; skip groups where p95_value is NaN
merged = trace_window.merge(trace_agg[['cmdb_id', 'trace_name', 'p95_value']],
                            on=['cmdb_id', 'trace_name'], how='left')
merged = merged.dropna(subset=['p95_value']).copy()
merged['p95_value'] = pd.to_numeric(merged['p95_value'], errors='coerce')

# Components of interest
components = ["MG01","MG02","dockerB1","IG01","Tomcat03","Tomcat04"]

# 3) Select rows for those components and where value > p95_value
comp_rows = merged[merged['cmdb_id'].isin(components)].copy()
comp_rows['is_anom'] = comp_rows['value'] > comp_rows['p95_value']
anoms = comp_rows[comp_rows['is_anom']].copy()

# 4) Prepare anomaly rows table with requested columns and sorting/limiting
if not anoms.empty:
    anoms['timestamp'] = anoms['ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    anomalies_table = anoms.sort_values('ts', ascending=True)[['timestamp','cmdb_id','trace_name','value','p95_value']].head(50).reset_index(drop=True)
else:
    # empty table with required columns
    anomalies_table = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95_value'])

# 5) Small aggregate per component: anomaly_count and earliest anomaly ts (for the listed components)
rows = []
for c in components:
    sub = anoms[anoms['cmdb_id'] == c]
    cnt = int(sub.shape[0])
    if cnt > 0:
        earliest = sub['ts'].min().strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        earliest = pd.NA
    rows.append((c, cnt, earliest))

comp_agg = pd.DataFrame(rows, columns=['cmdb_id','anomaly_count','earliest_ts'])

# Keep outputs compact
anomalies_table, comp_agg

```
Out[6]:
```
```python
# Summarize anomaly rows and per-component aggregates in plain English using existing variables:
# anomalies_table, comp_agg

# Reuse kernel variables
total_anom_rows = int(anomalies_table.shape[0])
time_range = "2021-03-10 04:30:00 — 2021-03-10 05:00:00 (UTC+8)"

# Overall earliest anomaly timestamp (if any)
earliest_overall = anomalies_table['timestamp'].min() if total_anom_rows > 0 else None

# Top components by anomaly count (from comp_agg)
top_components = comp_agg.sort_values('anomaly_count', ascending=False).head(5)

lines = [
    f"Trace anomalies within {time_range}:",
    f"- Total anomaly rows (selected components): {total_anom_rows}.",
]
if earliest_overall is not None:
    lines.append(f"- Earliest anomaly observed at: {earliest_overall}.")

lines.append("- Per-component summary (anomaly_count, earliest anomaly):")
for _, r in top_components.iterrows():
    earliest = r['earliest_ts'] if pd.notna(r['earliest_ts']) else "none"
    lines.append(f"  - {r['cmdb_id']}: count={int(r['anomaly_count'])}, earliest={earliest}")

lines.append(
    "\nInterpretation: Multiple services experienced short-lived trace metric breaches in the window. "
    "Highest anomaly counts are on Tomcat03 (13), MG02 (11), dockerB1 (11), and IG01 (9), with first anomalies "
    "appearing between ~04:35 and ~04:38 UTC+8. Many anomalies are in trace durations and row_count metrics, "
    "indicating brief increases in request load and elevated latencies between ~04:35–04:50."
)

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp   cmdb_id                      trace_name     value  p95_value
0   2021-03-10 04:35:00+0800      MG02        trace.self.duration_mean  0.067667   0.048850
1   2021-03-10 04:36:00+0800      MG02        trace.self.duration_mean  0.051400   0.048850
2   2021-03-10 04:37:00+0800      MG01        trace.self.duration_mean  0.074063   0.072467
3   2021-03-10 04:37:00+0800      MG01         trace.self.duration_p95  0.216500   0.214745
4   2021-03-10 04:37:00+0800  Tomcat04         trace.self.duration_p95  0.090400   0.088625
..                       ...       ...                             ...       ...        ...
45  2021-03-10 04:49:00+0800  Tomcat03         trace.self.duration_p95  0.142850   0.078097
46  2021-03-10 04:49:00+0800  Tomcat03        trace.self.duration_mean  0.022500   0.016629
47  2021-03-10 04:49:00+0800      MG02  trace.to_dockerB1.duration_p95  0.310000   0.131237
48  2021-03-10 04:49:00+0800      MG02         trace.self.duration_p95  0.164250   0.126850
49  2021-03-10 04:49:00+0800      IG01         trace.self.duration_p95  0.344000   0.316470

[50 rows x 5 columns],     cmdb_id  anomaly_count               earliest_ts
0      MG01              5  2021-03-10 04:37:00+0800
1      MG02             11  2021-03-10 04:35:00+0800
2  dockerB1             11  2021-03-10 04:38:00+0800
3      IG01              9  2021-03-10 04:38:00+0800
4  Tomcat03             13  2021-03-10 04:38:00+0800
5  Tomcat04              6  2021-03-10 04:37:00+0800)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import re

# Reuse existing trace_df from kernel
# Ensure trace_name is string
trace_df['trace_name'] = trace_df['trace_name'].astype(str)
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)

components = ["MG01","MG02","dockerB1","IG01","Tomcat03","Tomcat04"]

# Filter to components of interest (scan full trace_df)
trace_sub = trace_df[trace_df['cmdb_id'].isin(components)].copy()

# Group to get global counts per (cmdb_id, trace_name)
grp = trace_sub.groupby(['cmdb_id', 'trace_name']).size().reset_index(name='count_of_rows_in_trace_df')

# Extract inferred_peer from patterns 'trace.to_<PEER>' or 'trace.from_<PEER>'
# Pattern captures alphanumeric and hyphen/underscore in peer names
peer_pattern = r'trace\.(?:to|from)_([A-Za-z0-9_\-]+)'
grp['inferred_peer'] = grp['trace_name'].str.extract(peer_pattern, expand=False).fillna('')

# Sort by cmdb_id then trace_name and limit to top 200 rows
result_df = grp.sort_values(['cmdb_id','trace_name'], ascending=[True, True]).head(200).reset_index(drop=True)

# Display compact result
result_df

```
Out[7]:
```
```python
# Summarize the scan results for selected components using existing result_df in the kernel.
# Produce a compact plain-English summary string and display it.

import pandas as pd

# Reuse result_df computed earlier (unique trace_name rows for the specified components)
df = result_df.copy()

components = ["MG01","MG02","dockerB1","IG01","Tomcat03","Tomcat04"]

total_rows = df.shape[0]
lines = [
    "Trace name inventory (full trace_df) — selected components:",
    f"- Components scanned: {', '.join(components)}",
    f"- Total unique (cmdb_id, trace_name) rows found for these components: {total_rows}",
    ""
]

# Per-component compact summary
for c in components:
    sub = df[df['cmdb_id'] == c]
    n_trace_names = int(sub.shape[0])
    # inferred peers list and counts (only non-empty inferred_peer)
    peers = sub[sub['inferred_peer'] != ''].copy()
    if not peers.empty:
        peer_counts = peers.groupby('inferred_peer')['count_of_rows_in_trace_df'].sum().sort_values(ascending=False)
        peer_list = [f"{peer} ({int(cnt)})" for peer, cnt in peer_counts.items()]
        peer_preview = ", ".join(peer_list[:5])  # limit preview
        peer_info = f"{len(peer_counts)} peers -> {peer_preview}"
    else:
        peer_info = "no inferred peers"
    lines.append(f"- {c}: {n_trace_names} unique trace_names; {peer_info}")

lines.append("")
lines.append("Key observations (compact):")
# derive top service-to-service edges across these components
edges = df[df['inferred_peer'] != ''].groupby(['cmdb_id','inferred_peer'])['count_of_rows_in_trace_df'].sum().reset_index()
if edges.empty:
    lines.append("- No trace.to_/trace.from_ patterns found among these components.")
else:
    top_edges = edges.sort_values('count_of_rows_in_trace_df', ascending=False).head(8)
    lines.append("- Top inferred service-to-service relationships (component -> peer (row_count)):")
    for _, r in top_edges.iterrows():
        lines.append(f"  • {r['cmdb_id']} -> {r['inferred_peer']} ({int(r['count_of_rows_in_trace_df'])})")

lines.append("")
lines.append("Interpretation: The table shows which trace metrics exist per component and, when present,")
lines.append("extracts peers from trace.to_<PEER> or trace.from_<PEER> patterns to reveal downstream/upstream links.")
lines.append("Use the detailed result table (produced earlier) to inspect specific trace_name rows for each component.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  count_of_rows_in_trace_df inferred_peer
0        IG01         trace.self.duration_mean                         25              
1        IG01          trace.self.duration_p95                         25              
2        IG01             trace.self.row_count                         25              
3        IG01  trace.to_Tomcat01.duration_mean                         20      Tomcat01
4        IG01   trace.to_Tomcat01.duration_p95                         20      Tomcat01
..        ...                              ...                        ...           ...
133  dockerB1       trace.to_MG01.duration_p95                          9          MG01
134  dockerB1          trace.to_MG01.row_count                          9          MG01
135  dockerB1      trace.to_MG02.duration_mean                          9          MG02
136  dockerB1       trace.to_MG02.duration_p95                          9          MG02
137  dockerB1          trace.to_MG02.row_count                          9          MG02

[138 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing kernel variables: trace_df, trace_agg
tz = pytz.timezone('Asia/Shanghai')

# Ensure necessary columns and types
trace_df['trace_name'] = trace_df['trace_name'].astype(str)
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Ensure tz-aware timestamp column 'ts' exists and set to UTC+8
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # if existing, convert to tz-aware in Asia/Shanghai
    trace_df['ts'] = pd.to_datetime(trace_df['ts'])
    if not pd.api.types.is_datetime64tz_dtype(trace_df['ts']):
        trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        trace_df['ts'] = trace_df['ts'].dt.tz_convert(tz)

# 1) Filter to the given window (inclusive)
start_dt = tz.localize(pd.to_datetime('2021-03-10 04:30:00'))
end_dt = tz.localize(pd.to_datetime('2021-03-10 05:00:00'))
trace_window = trace_df[(trace_df['ts'] >= start_dt) & (trace_df['ts'] <= end_dt)].copy()

# 2) Join with global p95 from trace_agg and skip groups with NaN p95
merged = trace_window.merge(trace_agg[['cmdb_id','trace_name','p95_value']],
                            on=['cmdb_id','trace_name'], how='left')
merged = merged.dropna(subset=['p95_value']).copy()
merged['p95_value'] = pd.to_numeric(merged['p95_value'], errors='coerce')

# 3) Mark anomaly points where value > p95_value
merged['is_anom'] = merged['value'] > merged['p95_value']

# 4) Apply selection condition:
# - cmdb_id == 'MG01'
# - OR trace_name contains '_to_MG01' or '_from_MG01'
# - OR (cmdb_id == 'dockerB1' AND trace_name contains '_to_MG01' or '_from_MG01')
pattern = r'(_to_MG01|_from_MG01)'
name_has_mg01 = merged['trace_name'].str.contains(pattern, regex=True, na=False)
cond = (merged['cmdb_id'] == 'MG01') | name_has_mg01 | ((merged['cmdb_id'] == 'dockerB1') & name_has_mg01)

selected_anoms = merged[merged['is_anom'] & cond].copy()

# Prepare anomaly rows table (timestamp formatted, sorted)
if not selected_anoms.empty:
    selected_anoms['timestamp'] = selected_anoms['ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    anomalies_table = selected_anoms.sort_values('ts', ascending=True)[
        ['timestamp','cmdb_id','trace_name','value','p95_value']
    ].head(50).reset_index(drop=True)
else:
    anomalies_table = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95_value'])

# Aggregate per (cmdb_id, trace_name): anomaly_count and earliest_ts
if not selected_anoms.empty:
    agg = selected_anoms.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count=('is_anom','size'),
        earliest_ts=('ts','min')
    ).reset_index()
    agg['earliest_ts'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg_table = agg.sort_values(['anomaly_count','earliest_ts'], ascending=[False, True]).head(50).reset_index(drop=True)
else:
    agg_table = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_ts'])

# Return compact results
anomalies_table, agg_table

```
Out[8]:
```
```python
# Compose a concise plain-English summary based on existing anomalies_table and agg_table variables.
# Reuse kernel variables created in the previous step.

# Ensure variables exist
try:
    anomalies_table
    agg_table
except NameError:
    raise NameError("Required variables anomalies_table and agg_table not found in kernel.")

# Compute totals and per-group details
total_anomaly_rows = int(anomalies_table.shape[0])
groups = []
for _, r in agg_table.iterrows():
    cmdb = r['cmdb_id']
    tname = r['trace_name']
    cnt = int(r['anomaly_count'])
    earliest = r['earliest_ts']
    # compute max value and p95 from anomalies_table rows for this group (if present)
    subset = anomalies_table[(anomalies_table['cmdb_id'] == cmdb) & (anomalies_table['trace_name'] == tname)]
    if not subset.empty:
        max_value = float(subset['value'].max())
        p95_value = float(subset['p95_value'].iloc[0])
        severity = max_value / p95_value if p95_value != 0 else float('inf')
        groups.append((cmdb, tname, cnt, earliest, max_value, p95_value, round(severity, 3)))
    else:
        groups.append((cmdb, tname, cnt, earliest, None, None, None))

# Build summary lines
lines = []
lines.append("Trace anomalies focused on MG01-related paths (window 2021-03-10 04:30–05:00 UTC+8):")
lines.append(f"- Total anomaly rows matching selection: {total_anomaly_rows}.")
if groups:
    lines.append("- Per-group summary (cmdb_id | trace_name — count, earliest, max_value, p95_value, severity_ratio):")
    for cmdb, tname, cnt, earliest, maxv, p95v, sev in groups:
        if maxv is not None:
            lines.append(f"  • {cmdb} | {tname} — {cnt} anomaly(ies), earliest={earliest}, max={maxv}, p95={p95v}, severity={sev}")
        else:
            lines.append(f"  • {cmdb} | {tname} — {cnt} anomaly(ies), earliest={earliest}")
else:
    lines.append("- No matching anomaly groups found.")

lines.append("")
lines.append("Key takeaway: MG01 itself shows multiple short-lived breaches (duration_mean, duration_p95, row_count) around 04:37–04:39 UTC+8. "
             "The largest observed breach in the selected set is MG01 row_count = 60 vs p95 = 44 and duration_p95 spikes (e.g., 0.4894 vs 0.2147), "
             "indicating brief but notable latency and traffic increases on MG01 and its related traces.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp cmdb_id                trace_name      value  p95_value
0  2021-03-10 04:37:00+0800    MG01  trace.self.duration_mean   0.074063   0.072467
1  2021-03-10 04:37:00+0800    MG01   trace.self.duration_p95   0.216500   0.214745
2  2021-03-10 04:38:00+0800    MG01      trace.self.row_count  60.000000  44.000000
3  2021-03-10 04:39:00+0800    MG01  trace.self.duration_mean   0.079435   0.072467
4  2021-03-10 04:39:00+0800    MG01   trace.self.duration_p95   0.489400   0.214745,   cmdb_id                trace_name  anomaly_count               earliest_ts
0    MG01  trace.self.duration_mean              2  2021-03-10 04:37:00+0800
1    MG01   trace.self.duration_p95              2  2021-03-10 04:37:00+0800
2    MG01      trace.self.row_count              1  2021-03-10 04:38:00+0800)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing kernel variables: metric_df, metric_agg
tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric and timestamp tz-aware column 'ts' for metric_df (create if missing or not tz-aware)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # If exists but not tz-aware or not Asia/Shanghai, recreate from raw timestamp to be safe
    try:
        if not pd.api.types.is_datetime64tz_dtype(metric_df['ts']):
            metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            metric_df['ts'] = metric_df['ts'].dt.tz_convert(tz)
    except Exception:
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define window (inclusive) in UTC+8
start_dt = tz.localize(pd.to_datetime('2021-03-10 04:30:00'))
end_dt = tz.localize(pd.to_datetime('2021-03-10 05:00:00'))

# 1) Filter metric_df to MG01 and the time window
mg_window = metric_df[(metric_df['cmdb_id'] == 'MG01') & (metric_df['ts'] >= start_dt) & (metric_df['ts'] <= end_dt)].copy()

# 2) Join with global metric_agg to attach p95_value (metric_agg computed globally earlier)
# Ensure metric_agg has required columns
if 'p95_value' not in metric_agg.columns:
    raise KeyError("metric_agg missing 'p95_value' column (global thresholds must be computed before filtering).")

mg_merged = mg_window.merge(metric_agg[['cmdb_id','kpi_name','p95_value']],
                            on=['cmdb_id','kpi_name'], how='left')

# Skip groups where p95_value is NaN (per instructions)
mg_merged = mg_merged.dropna(subset=['p95_value']).copy()
mg_merged['p95_value'] = pd.to_numeric(mg_merged['p95_value'], errors='coerce')

# 3) Identify anomaly rows where value > p95_value
mg_merged['is_anom'] = mg_merged['value'] > mg_merged['p95_value']

# 4) Aggregate per (cmdb_id, kpi_name) for MG01 where anomaly_count > 0
anom_rows = mg_merged[mg_merged['is_anom']].copy()

if anom_rows.empty:
    # Empty results: prepare empty frames with required columns
    mg_metric_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value'])
    mg_metric_anoms = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','p95_value'])
else:
    agg = anom_rows.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count=('is_anom','size'),
        earliest_ts=('ts','min'),
        latest_ts=('ts','max'),
        max_value=('value','max')
    ).reset_index()
    # attach p95_value per group (global)
    p95_per_group = anom_rows.groupby(['cmdb_id','kpi_name'])['p95_value'].first().reset_index()
    mg_metric_agg = agg.merge(p95_per_group, on=['cmdb_id','kpi_name'], how='left')
    # format timestamps compactly
    mg_metric_agg['earliest_ts'] = mg_metric_agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    mg_metric_agg['latest_ts'] = mg_metric_agg['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    # sort and limit top 20
    mg_metric_agg = mg_metric_agg[['cmdb_id','kpi_name','anomaly_count','earliest_ts','latest_ts','max_value','p95_value']] \
        .sort_values(['anomaly_count','max_value'], ascending=[False, False]).head(20).reset_index(drop=True)

    # 5) Prepare raw anomaly rows table (timestamp, cmdb_id, kpi_name, value, p95_value), sorted by timestamp asc and limited to 50
    anom_rows['timestamp'] = anom_rows['ts'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    mg_metric_anoms = anom_rows.sort_values('ts', ascending=True)[
        ['timestamp','cmdb_id','kpi_name','value','p95_value']
    ].head(50).reset_index(drop=True)

# Return the compact aggregate and the raw anomaly rows (as two variables)
mg_metric_agg, mg_metric_anoms

```
Out[9]:
```
```python
# Compose a compact plain-English summary using existing kernel variables mg_metric_agg and mg_metric_anoms

try:
    mg_metric_agg
    mg_metric_anoms
except NameError:
    raise NameError("Required variables mg_metric_agg and mg_metric_anoms not found in kernel.")

groups_count = int(mg_metric_agg.shape[0])
raw_rows = int(mg_metric_anoms.shape[0])

lines = [
    "MG01 metric anomalies (window 2021-03-10 04:30–05:00 UTC+8):",
    f"- Number of MG01 metric groups with anomalies: {groups_count}",
    f"- Total anomaly rows for MG01 in window: {raw_rows}",
]

# summarize top groups (already limited to top 20)
for _, r in mg_metric_agg.iterrows():
    lines.append(
        f"  • {r['kpi_name']} — anomalies={int(r['anomaly_count'])}, "
        f"earliest={r['earliest_ts']}, latest={r['latest_ts']}, "
        f"max_value={r['max_value']:.6g}, p95={r['p95_value']:.6g}"
    )

lines.append("")
lines.append("Interpretation: The only detected MG01 metric anomaly in this window is a short JVM heap usage spike "
             "('JVM-Memory_7779_JVM_Memory_HeapMemoryUsed') with two points at 04:59–05:00 exceeding the global P95 "
             "(max ~1.0436e+09 vs p95 ~1.03898e+09). This indicates a brief heap increase near the window end and may warrant investigation for potential memory pressure but is not a sustained OOM pattern in this window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                   kpi_name  anomaly_count               earliest_ts                 latest_ts     max_value     p95_value
0    MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed              2  2021-03-10 04:59:00+0800  2021-03-10 05:00:00+0800  1.043554e+09  1.038978e+09,                   timestamp cmdb_id                                   kpi_name         value     p95_value
0  2021-03-10 04:59:00+0800    MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.039980e+09  1.038978e+09
1  2021-03-10 05:00:00+0800    MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.043554e+09  1.038978e+09)```
```